In [3]:
# Install required dependencies
%pip install opencv-python numpy mediapipe==0.10.14

# Create required directories for webcam data collection
import os
os.makedirs("webcam_data", exist_ok=True)
os.makedirs("processed_webcam_data/landmarks", exist_ok=True)
os.makedirs("processed_webcam_data/preview", exist_ok=True)
print("Required directories created!")

  Using cached protobuf-4.25.9-cp310-abi3-win_amd64.whl.metadata (541 bytes)
  Using cached ml_dtypes-0.5.4-cp310-cp310-win_amd64.whl.metadata (9.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached scipy-1.15.3-cp310-cp310-win_amd64.whl.metadata (60 kB)
   ---------------------------------------- 0.0/50.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/50.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/50.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/50.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/50.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/50.8 MB ? eta -:--:--
   ---------------------------------------- 0.5/50.8 MB 645.7 kB/s eta 0:01:18
    --------------------------------------- 0.8/50.8 MB 907.1 kB/s eta 0:00:56
    --------------------------------------- 0.8/50.8 MB 907.1 kB/s eta 0:00:56
    --------------------------------------- 1.0/50.8 

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'd:\\BRIDGING SILENCE DATA COLLECTION PIPELINE\\.venv\\Lib\\site-packages\\scipy\\cluster\\_hierarchy.cp310-win_amd64.pyd'
Check the permissions.



  Using cached mediapipe-0.10.14-cp310-cp310-win_amd64.whl.metadata (9.9 kB)
  Using cached jaxlib-0.6.2-cp310-cp310-win_amd64.whl.metadata (1.4 kB)
  Using cached protobuf-4.25.9-cp310-abi3-win_amd64.whl.metadata (541 bytes)
  Using cached ml_dtypes-0.5.4-cp310-cp310-win_amd64.whl.metadata (9.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached scipy-1.15.3-cp310-cp310-win_amd64.whl.metadata (60 kB)
   ---------------------------------------- 0.0/50.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/50.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/50.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/50.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/50.8 MB ? eta -:--:--
   ---------------------------------------- 0.5/50.8 MB 1.2 MB/s eta 0:00:43
    --------------------------------------- 0.8/50.8 MB 1.2 MB/s eta 0:00:42
   - -------------------------------------- 1.3/50.8 MB 1

In [1]:
# Core data processing functions matching the main pipeline format
import cv2
import numpy as np
import mediapipe as mp
from pathlib import Path

MAX_SEQ_LENGTH = 120
FEATURES = 141
FPS = 30
WIDTH = 640
HEIGHT = 480

mp_holistic = mp.solutions.holistic
mp_draw = mp.solutions.drawing_utils

POSE = [0, 11, 12, 13, 14]

def normalize_frame(vec):
    vec = vec.copy()
    pose = vec[:15].reshape(5, 3)
    lh = vec[15:78].reshape(21, 3)
    rh = vec[78:].reshape(21, 3)

    if np.any(pose):
        ls = pose[1]
        rs = pose[2]
        center = (ls + rs) / 2
        dist = np.linalg.norm(ls - rs)
        if dist > 1e-6:
            pose = (pose - center) / dist
            if np.any(lh): lh = (lh - center) / dist
            if np.any(rh): rh = (rh - center) / dist

    return np.concatenate([pose.flatten(), lh.flatten(), rh.flatten()])

def extract(results):
    pose = []
    if results.pose_landmarks:
        for idx in POSE:
            lm = results.pose_landmarks.landmark[idx]
            pose.extend([lm.x, lm.y, lm.z])
    else: pose = [0] * 15

    lh = []
    if results.left_hand_landmarks:
        for lm in results.left_hand_landmarks.landmark:
            lh.extend([lm.x, lm.y, lm.z])
    else: lh = [0] * 63

    rh = []
    if results.right_hand_landmarks:
        for lm in results.right_hand_landmarks.landmark:
            rh.extend([lm.x, lm.y, lm.z])
    else: rh = [0] * 63

    return normalize_frame(np.array(pose + lh + rh, dtype=np.float32))

def pad(seq):
    if len(seq) < MAX_SEQ_LENGTH:
        seq = np.vstack([seq, np.zeros((MAX_SEQ_LENGTH - len(seq), FEATURES))])
    return seq[:MAX_SEQ_LENGTH]

def reconstruct(data, outfile):
    writer = cv2.VideoWriter(str(outfile), cv2.VideoWriter_fourcc(*"mp4v"), FPS, (WIDTH, HEIGHT))
    for i, frame in enumerate(data):
        canvas = np.full((HEIGHT, WIDTH, 3), 255, dtype=np.uint8)
        if np.all(frame == 0):
            cv2.putText(canvas, "PAD", (280, 220), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 2)
            writer.write(canvas)
            continue

        pose = frame[:15].reshape(5, 3)
        lh = frame[15:78].reshape(21, 3)
        rh = frame[78:].reshape(21, 3)

        def draw(points, color):
            for p in points:
                if p[0] == 0 and p[1] == 0 and p[2] == 0:
                    continue
                x = int((p[0] * 0.25 + 0.5) * WIDTH)
                y = int((p[1] * 0.25 + 0.5) * HEIGHT)
                cv2.circle(canvas, (x, y), 3, color, -1)

        draw(pose, (0, 0, 255))
        draw(lh, (0, 255, 0))
        draw(rh, (255, 255, 0))

        cv2.putText(canvas, f"Frame {i}", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, .7, (0, 0, 0), 2)
        writer.write(canvas)
    writer.release()

In [2]:
import time

WEBCAM_DATA_DIR = "webcam_data"
WEBCAM_OUT_DIR = "processed_webcam_data"

def collect_webcam_data(word, num_samples=1, frames_per_sample=90):
    raw_root = Path(WEBCAM_DATA_DIR) / word
    land_root = Path(WEBCAM_OUT_DIR) / "landmarks" / word
    prev_root = Path(WEBCAM_OUT_DIR) / "preview" / word
    
    raw_root.mkdir(parents=True, exist_ok=True)
    land_root.mkdir(parents=True, exist_ok=True)
    prev_root.mkdir(parents=True, exist_ok=True)
    
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return

    with mp_holistic.Holistic(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holo:
        for sample_idx in range(num_samples):
            # Countdown
            for countdown in [3, 2, 1]:
                start_time = time.time()
                while time.time() - start_time < 1.0:
                    ret, frame = cap.read()
                    if not ret: break
                    frame = cv2.resize(frame, (WIDTH, HEIGHT))
                    cv2.putText(frame, f"Starting in {countdown}...", (50, 50), 
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                    cv2.imshow('Data Collection', frame)
                    cv2.waitKey(1)
            
            print(f"Action! Recording {word} (sample {sample_idx+1})")
            
            frames = []
            seq = []
            
            for frame_num in range(frames_per_sample):
                ret, frame = cap.read()
                if not ret:
                    break
                
                frame = cv2.resize(frame, (WIDTH, HEIGHT))
                
                display_frame = frame.copy()
                cv2.putText(display_frame, f"Recording: {word} [{frame_num}/{frames_per_sample}]", 
                            (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                cv2.imshow('Data Collection', display_frame)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
                
                frames.append(frame)
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                res = holo.process(rgb)
                seq.append(extract(res))
                
            sample_name = f"sample_{int(time.time())}"
            raw_vid_path = raw_root / f"{sample_name}.mp4"
            
            writer = cv2.VideoWriter(str(raw_vid_path), cv2.VideoWriter_fourcc(*"mp4v"), FPS, (WIDTH, HEIGHT))
            for f in frames: writer.write(f)
            writer.release()
            
            seq = np.array(seq)
            original = len(seq)
            seq = pad(seq)
            
            npy_path = land_root / f"{sample_name}.npy"
            np.save(npy_path, seq)
            
            prev_path = prev_root / f"{sample_name}.mp4"
            reconstruct(seq, prev_path)
            
            print(f"✓ Saved {sample_name} | frames={original} -> {seq.shape}")
            
            if sample_idx < num_samples - 1:
                cv2.waitKey(1000)
            
    cap.release()
    cv2.destroyAllWindows()
    print("\nCollection complete!")

# -----------------------------------------------
# Example usage: Change "HELLO" to the word you want to collect
# -----------------------------------------------
collect_webcam_data("HELLO", num_samples=3, frames_per_sample=90)

Action! Recording HELLO (sample 1)


d:\BRIDGING SILENCE DATA COLLECTION PIPELINE\.venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


✓ Saved sample_1782131548 | frames=90 -> (161, 141)
Action! Recording HELLO (sample 2)
✓ Saved sample_1782131561 | frames=90 -> (161, 141)
Action! Recording HELLO (sample 3)
✓ Saved sample_1782131574 | frames=90 -> (161, 141)

Collection complete!
